# Predicting late delivery
Question: can we predict `late_delivery` at dispatch time? Each consignment is one row, and a model would be used on later dispatches, so the split is temporal.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.consignment.is_unique, df.late_delivery.mean())

## Feature policy (checklist 7)
`columns.csv`: `claim_paid_eur` is compensation on a claim settled *after* delivery, so it is not known at dispatch (it is a consequence of lateness) and is dropped. `consignment` is an identifier and is dropped. `dispatched_on` is used only to order/split and to derive hour and weekday (known at dispatch); no absolute date/trend feature is used, since the rate drifts strongly over time (below).

In [ ]:
print(df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.mean().round(3).to_string())

## Test set (checklist 1)
The model is for future dispatches and each consignment appears once, so the test set is the latest 20% of rows by dispatch time. It is not touched until the final step.

In [ ]:
df['hour'] = df.dispatched_on.dt.hour; df['dow'] = df.dispatched_on.dt.dayofweek
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min(), len(train), len(test))
print('late rate train/test', train.late_delivery.mean(), test.late_delivery.mean())
FEATS = ['carrier','distance_km','weight_kg','parcels','hour','dow']
Xtr, ytr = train[FEATS], train.late_delivery.values

## Candidates (checklist 4), named before scoring
1. **Baseline**: predicts the training prior (no features).
2. **Logistic regression** (scaled numerics + one-hot carrier), C in {0.1, 1}.
3. **Gradient boosting** (HistGradientBoosting), max_depth in {3, 6}, learning_rate 0.05.

Selection metric: mean ROC AUC over expanding-window CV folds within the training rows. Decision threshold: chosen on pooled out-of-fold predictions of the chosen model to maximise F1. Preprocessing sits inside the Pipeline, so it is fitted per fold (checklist 3).

In [ ]:
num = ['distance_km','weight_kg','parcels','hour','dow']
def pre(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier']),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
cands = {'baseline': Pipeline([('p', pre(False)), ('m', DummyClassifier(strategy='prior'))])}
for C_ in (0.1, 1):
    cands[f'logreg_C{C_}'] = Pipeline([('p', pre(True)), ('m', LogisticRegression(C=C_, max_iter=1000))])
for d in (3, 6):
    cands[f'hgb_depth{d}'] = Pipeline([('p', pre(False)), ('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=0.05, max_iter=200, random_state=0))])
tscv = TimeSeriesSplit(n_splits=5)
oof, res = {}, []
for name, pipe in cands.items():
    aucs, pr, idx, preds = [], [], [], []
    for tr, va in tscv.split(Xtr):
        m = clone(pipe).fit(Xtr.iloc[tr], ytr[tr])
        p = m.predict_proba(Xtr.iloc[va])[:,1]
        aucs.append(roc_auc_score(ytr[va], p)); pr.append(average_precision_score(ytr[va], p))
        idx.append(va); preds.append(p)
    oof[name] = (np.concatenate(idx), np.concatenate(preds))
    res.append((name, np.mean(aucs), np.std(aucs), np.mean(pr)))
cv = pd.DataFrame(res, columns=['model','cv_auc','auc_sd','cv_pr_auc']).set_index('model'); cv

In [ ]:
best = cv.cv_auc.idxmax(); print('chosen:', best)
idx, p = oof[best]; y = ytr[idx]
ths = np.linspace(0.05, 0.8, 76)
f1s = [f1_score(y, p>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))]); print('threshold', thr)

## Decision (checklist 5), recorded before test scoring
The model and threshold are those printed above, chosen from CV only. The next cell refits on all training rows and scores the test set once.

In [ ]:
final = clone(cands[best]).fit(Xtr, ytr)
pt = final.predict_proba(test[FEATS])[:,1]; yt = test.late_delivery.values
base = clone(cands['baseline']).fit(Xtr, ytr).predict_proba(test[FEATS])[:,1]
print(f'TEST {best}: ROC AUC {roc_auc_score(yt,pt):.3f}, PR AUC {average_precision_score(yt,pt):.3f}, F1@{thr:.2f} {f1_score(yt,pt>=thr):.3f}')
print(f'test prevalence {yt.mean():.3f}; baseline ROC AUC {roc_auc_score(yt,base):.3f}, PR AUC {average_precision_score(yt,base):.3f}')

## Conclusion
The printed test score above is the headline (single evaluation). Caveat: the late rate falls sharply in the last months, so test prevalence differs from training, which affects F1/threshold and precision-recall numbers more than AUC. Only dispatch-time features were used.